# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [15]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [14]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Add uv to PATH for this notebook session
# import os
# os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

# # Check uv works
# !uv --version

# # Create venv
# !uv venv .venv --seed

# # Install dependencies into the venv
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart kernel, then select: Python (cse151b)")

### Run the cell below every time to activate the installed environment. 

In [1]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

In [1]:
# import sys
# print(sys.executable)

/workspace/151B_SP26_Competition/.venv/bin/python


In [13]:
# import sys, subprocess

# pkgs = ["torch", "transformers", "vllm", "bitsandbytes"]

# for pkg in pkgs:
#     print(f"\nTesting {pkg}...")
#     result = subprocess.run(
#         [sys.executable, "-c", f"import {pkg}; print('{pkg} OK')"],
#         capture_output=True,
#         text=True,
#         timeout=60,
#     )
#     print(result.stdout)
#     print(result.stderr)

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
print("Task started...")

import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

print("Task complete!")

Task started...
Task complete!


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
print("Task started...")

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

print("Task complete!")

Task started...
Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}
Task complete!


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [16]:
# import sys

# !{sys.executable} -m pip uninstall -y vllm
# !{sys.executable} -m pip install "vllm==0.19.1"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=2151) INFO 05-03 03:22:04 [weight_utils.py:581] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 20.987014 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=2151) INFO 05-03 03:22:06 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 37.523080 seconds
(EngineCore pid=2151) INFO 05-03 03:22:38 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/8c9f128ca0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2151) INFO 05-03 03:22:38 [backends.py:1111] Dynamo bytecode transform time: 30.80 s
(EngineCore pid=2151) INFO 05-03 03:22:47 [backends.py:372] Cache the graph of compile range (1, 32768) for later use
(EngineCore pid=2151) INFO 05-03 03:22:53 [backends.py:390] Compiling a graph for compile range (1, 32768) takes 15.23 s
(EngineCore pid=2151) INFO 05-03 03:23:06 [decorators.py:655] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/8c300eb6da3e0ead728b359a7a6320e1b3efb48edaff76c86977facba6772916/rank_0_0/model
(EngineCore pid=2151) INFO 05-03 03:23:06 [monitor.py:48] torch.compile took 58.83 s in total
(EngineCore pid=2151) INFO 05-03 03:

(EngineCore pid=2151) 2026-05-03 03:23:16,315 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2151) 2026-05-03 03:23:16,408 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.85it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 14.66it/s]


(EngineCore pid=2151) INFO 05-03 03:23:24 [gpu_model_runner.py:6046] Graph capturing finished in 8 secs, took 0.70 GiB
(EngineCore pid=2151) INFO 05-03 03:23:24 [gpu_worker.py:597] CUDA graph pool memory: 0.7 GiB (actual), 0.49 GiB (estimated), difference: 0.21 GiB (29.3%).
(EngineCore pid=2151) INFO 05-03 03:23:24 [core.py:283] init engine (profile, create kv cache, warmup model) took 77.43 seconds
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [17]:
# import sys
# !{sys.executable} -m pip install -U accelerate

In [6]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [7]:
# Build prompts for first 5 entries MODIFIED TO ALL
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Rendering prompts:   0%|          | 0/1126 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1126 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by making sure I know what the first few even positive whole numbers are. Positive even whole numbers start at 2, right? So the first one is 2, second is 4, third is 6, fourth  ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to check if the integral converges. Since the integrand is even (because s² is even, so the function is even in s), we can write it as 2 times the integral from 0 to infinity of (a^(3/2))/(s² + a²) ds. That might help.

But first, let's recall th ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about the turkey cooling down, so I think we

### Generate with Transformers (for Datahub)

In [18]:
# from transformers import TextStreamer

# import torch
# print(torch.cuda.is_available(), "\n")
# print(torch.cuda.get_device_name(0), "\n")
# print(next(llm.parameters()).device, "\n")

# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [
#             {"role": "system", "content": system},
#             {"role": "user", "content": user},
#         ],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# responses = []

# for i, prompt in enumerate(prompts):
#     print(f"\n── Generating Response {i} (id={data[i].get('id')}) ──")

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=16384,
#     ).to(llm.device)

#     streamer = TextStreamer(
#         tokenizer,
#         skip_prompt=True,
#         skip_special_tokens=True,
#     )

#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             repetition_penalty=1.0,
#             do_sample=True,
#             streamer=streamer,
#         )

#     # Decode only new tokens
#     new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
#     response = tokenizer.decode(
#         new_tokens,
#         skip_special_tokens=True,
#     ).strip()

#     responses.append(response)

#     print(f"\n── Finished Response {i} ──")

In [19]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.inference_mode():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         do_sample=True,
#         repetition_penalty=1.0,
#         pad_token_id=tokenizer.eos_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [8]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 1126/1126 [01:33<00:00, 12.07it/s]

Scoring complete. 1126 results.


## 8. Summary

Print accuracy broken down by question type.

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  276 /  375  (73.60%)
  Free-form  :  416 /  751  (55.39%)
  Overall    :  692 / 1126  (61.46%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [10]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1126 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [1]:
import sys
print(sys.executable)
# Expect "/workspace/151B_SP26_Competition/.venv/bin/python"

/workspace/151B_SP26_Competition/.venv/bin/python


In [8]:
print("Cell 1: imports, config, data, prompts...")

import os
import re
import sys
import gc
import json
import time
import shutil
import subprocess
from pathlib import Path
from typing import Optional, Dict, List, Tuple
from collections import Counter, defaultdict

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# ── Main run config ─────────────────────────────────────────────
RUN_NAME = "vllm_vote_run1"   # change this for a new run

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/public.jsonl"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESPONSE_PATH = RESULTS_DIR / f"{RUN_NAME}_responses.jsonl"
ATTEMPT_PATH = RESULTS_DIR / f"{RUN_NAME}_attempts.jsonl"
VOTE_PATH = RESULTS_DIR / f"{RUN_NAME}_votes.jsonl"
EVAL_PATH = RESULTS_DIR / f"{RUN_NAME}_eval.jsonl"
SUMMARY_PATH = RESULTS_DIR / f"{RUN_NAME}_summary.json"

# ── GPU config ─────────────────────────────────────────────────
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

# ── vLLM/model config ──────────────────────────────────────────
# Need this large enough for retry prompts + 81920 output tokens.
MAX_MODEL_LEN = 131072

GPU_MEMORY_UTILIZATION = 0.8
MAX_NUM_SEQS = 512
MAX_NUM_BATCHED_TOKENS = 262144

# ── Voting / generation config ─────────────────────────────────
DEFAULT_MAX_TOKENS = 32768
RETRY_MAX_TOKENS = 81920

NUM_VOTES = 5
AGREE_THRESHOLD = 3
MAX_RETRIES = 3

# Outer chunks for checkpointing. vLLM handles internal batching.
BATCH_SIZE = 256
RETRY_BATCH_SIZE = 32

# If still unresolved after retries, save best available answer anyway.
SAVE_BEST_AFTER_MAX_RETRIES = True

# Limit for testing. Use None for full dataset.
TEST_LIMIT = None

print(f"RUN_NAME                  = {RUN_NAME}")
print(f"RESPONSE_PATH             = {RESPONSE_PATH}")
print(f"ATTEMPT_PATH              = {ATTEMPT_PATH}")
print(f"VOTE_PATH                 = {VOTE_PATH}")
print(f"EVAL_PATH                 = {EVAL_PATH}")
print(f"MAX_MODEL_LEN             = {MAX_MODEL_LEN}")
print(f"DEFAULT_MAX_TOKENS        = {DEFAULT_MAX_TOKENS}")
print(f"RETRY_MAX_TOKENS          = {RETRY_MAX_TOKENS}")
print(f"NUM_VOTES                 = {NUM_VOTES}")
print(f"AGREE_THRESHOLD           = {AGREE_THRESHOLD}")
print(f"MAX_RETRIES               = {MAX_RETRIES}")
print(f"BATCH_SIZE                = {BATCH_SIZE}")
print(f"GPU_MEMORY_UTILIZATION    = {GPU_MEMORY_UTILIZATION}")

# ── Load data ──────────────────────────────────────────────────
t0 = time.time()

with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f]

if TEST_LIMIT is not None:
    data = data[:TEST_LIMIT]

n_mcq = sum(bool(d.get("options")) for d in data)
n_frq = len(data) - n_mcq

print(f"Loaded {len(data)} questions: {n_mcq} MCQ, {n_frq} FRQ")
print(f"Data loading took {time.time() - t0:.2f}s")

# ── Prompts ────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an expert competition mathematician. "
    "Solve the problem carefully and efficiently. "
    "Check for traps, arithmetic errors, algebraic mistakes, domain restrictions, "
    "and whether the final answer matches the requested format. "
    "At the end, output exactly one final answer in \\boxed{...}. "
    "For multiple-choice questions, the box must contain only the chosen letter. "
    "For free-response questions with multiple parts, include all requested answers "
    "inside one box, separated by commas, preserving labels if useful. "
    "Do not write anything after the boxed answer."
)

RETRY_SYSTEM_PROMPT = (
    "You are an expert competition mathematician re-evaluating a problem after "
    "several previous solution attempts disagreed or failed to produce a complete answer. "
    "Use the previous attempts as evidence, but do not blindly trust them. "
    "Identify likely mistakes, recompute carefully, and produce one final answer. "
    "Your final line must be exactly one \\boxed{...}. "
    "Do not write anything after the boxed answer."
)

def build_problem_text(item: dict) -> str:
    question = item["question"]

    if item.get("options"):
        labels = [chr(65 + i) for i in range(len(item["options"]))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, item["options"])
        )
        return (
            f"{question}\n\n"
            f"Options:\n{opts_text}\n\n"
            "Select the single best option. End with exactly one boxed letter, e.g. \\boxed{C}."
        )

    return (
        f"{question}\n\n"
        "Solve carefully. If there are multiple parts or multiple requested values, answer all of them. "
        "End with exactly one boxed final answer."
    )

def make_initial_prompt(item: dict) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_problem_text(item)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def normalize_answer_key(ans: str, is_mcq: bool) -> str:
    ans = ans.strip()

    if is_mcq:
        m = re.search(r"[A-Za-z]", ans)
        return m.group(0).upper() if m else ""

    ans = ans.lower()
    ans = re.sub(r"\s+", "", ans)
    ans = ans.replace("\\left", "").replace("\\right", "")
    return ans

def extract_boxed(text: str) -> str:
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text, flags=re.DOTALL)
    return matches[-1].strip() if matches else ""

def extract_answer_key(text: str, is_mcq: bool) -> str:
    boxed = extract_boxed(text)

    if boxed:
        return normalize_answer_key(boxed, is_mcq)

    if is_mcq:
        # JSON-ish fallback
        m = re.search(r'"answer"\s*:\s*"([A-Za-z])"', text)
        if m:
            return m.group(1).upper()

        # Last standalone letter fallback
        matches = re.findall(r"\b([A-Z])\b", text.upper())
        return matches[-1] if matches else ""

    return ""

def has_complete_answer(text: str, is_mcq: bool) -> bool:
    key = extract_answer_key(text, is_mcq)
    if not key:
        return False

    if is_mcq:
        return bool(re.fullmatch(r"[A-Z]", key))

    return bool(extract_boxed(text))

def format_eta(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.0f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.1f}h"

def truncate_text(s: str, max_chars: int = 1200) -> str:
    s = s.strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + "\n...[truncated]..."

print("Cell 1 complete.")

Cell 1: imports, config, data, prompts...
RUN_NAME                  = vllm_vote_run1
RESPONSE_PATH             = results/vllm_vote_run1_responses.jsonl
ATTEMPT_PATH              = results/vllm_vote_run1_attempts.jsonl
VOTE_PATH                 = results/vllm_vote_run1_votes.jsonl
EVAL_PATH                 = results/vllm_vote_run1_eval.jsonl
MAX_MODEL_LEN             = 131072
DEFAULT_MAX_TOKENS        = 32768
RETRY_MAX_TOKENS          = 81920
NUM_VOTES                 = 5
AGREE_THRESHOLD           = 3
MAX_RETRIES               = 3
BATCH_SIZE                = 256
GPU_MEMORY_UTILIZATION    = 0.8
Loaded 1126 questions: 375 MCQ, 751 FRQ
Data loading took 0.02s
Cell 1 complete.


In [9]:
print("Cell 2: loading tokenizer and vLLM model...")

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded in {time.time() - t0:.2f}s")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nCurrent nvidia-smi before model load:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("Could not run nvidia-smi:", e)

t0 = time.time()

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=MAX_NUM_SEQS,
    max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
)

print(f"Model loaded in {(time.time() - t0) / 60:.2f} min.")

print("\nCurrent nvidia-smi after model load:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("Could not run nvidia-smi:", e)

print("Cell 2 complete.")

Cell 2: loading tokenizer and vLLM model...
Tokenizer loaded in 0.94s
CUDA available: True
GPU: NVIDIA H100 80GB HBM3

Current nvidia-smi before model load:
Sun May  3 12:42:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:9A:00.0 Off |                    0 |
| N/A   26C    P0             69W /  700W |       4MiB /  81559MiB |      0

(EngineCore pid=125373) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=125373) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=125373) INFO 05-03 12:44:59 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.03it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.31it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  1.83it/s]
(EngineCore pid=125373) 


(EngineCore pid=125373) INFO 05-03 12:45:02 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 15.387541 seconds
(EngineCore pid=125373) INFO 05-03 12:45:28 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/63c09c1d7f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=125373) INFO 05-03 12:45:28 [backends.py:1111] Dynamo bytecode transform time: 26.38 s
(EngineCore pid=125373) INFO 05-03 12:45:31 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 262144) from the cache, took 2.014 s
(EngineCore pid=125373) INFO 05-03 12:45:31 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/0fcc80e4cab569d27613d2efad5353a28970fa742ad0b02d296e2e4a607831ea/rank_0_0/model
(EngineCore pid=125373) INFO 05-03 12:45:31 [monitor.py:48] torch.compile took 28.73 s in total
(EngineCore pid=125373) INFO 05-03 12:45:33 [monitor.py:76] Initial profiling/warmup run took 1.98 

(EngineCore pid=125373) 2026-05-03 12:45:44,108 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=125373) 2026-05-03 12:45:45,843 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 11.55it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:03<00:00, 15.90it/s]


(EngineCore pid=125373) INFO 05-03 12:45:55 [gpu_model_runner.py:6046] Graph capturing finished in 10 secs, took 0.89 GiB
(EngineCore pid=125373) INFO 05-03 12:45:55 [gpu_worker.py:597] CUDA graph pool memory: 0.89 GiB (actual), 0.64 GiB (estimated), difference: 0.25 GiB (27.9%).
(EngineCore pid=125373) INFO 05-03 12:45:55 [core.py:283] init engine (profile, create kv cache, warmup model) took 53.32 seconds
(EngineCore pid=125373) INFO 05-03 12:45:57 [vllm.py:790] Asynchronous scheduling is enabled.
Model loaded in 3.60 min.

Current nvidia-smi after model load:
Sun May  3 12:45:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr

In [10]:
print("Cell 3: majority-vote generation with checkpointing...")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Using response file:", RESPONSE_PATH)

THINK_END_TOKEN_ID = 151668

def extract_qwen_final_content_from_vllm(completion, raw_text: str):
    token_ids = getattr(completion, "token_ids", None)

    if token_ids is not None:
        ids = list(token_ids)

        try:
            idx = len(ids) - ids[::-1].index(THINK_END_TOKEN_ID)
            final_ids = ids[idx:]
            final_text = tokenizer.decode(final_ids, skip_special_tokens=True).strip()
            if final_text:
                return final_text, True
        except ValueError:
            pass

    if "</think>" in raw_text:
        return raw_text.split("</think>")[-1].strip(), True

    return raw_text.strip(), False

def make_sampling_params(max_tokens: int, n: int) -> SamplingParams:
    return SamplingParams(
        n=n,
        max_tokens=max_tokens,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
    )

# ── Load completed responses ───────────────────────────────────
responses_by_id = {}

if RESPONSE_PATH.exists():
    with open(RESPONSE_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            responses_by_id[str(row["id"])] = row

print(f"Already accepted: {len(responses_by_id)}/{len(data)}")

# ── Load prior attempts, so retries can reference previous runs ─
attempts_by_id = defaultdict(list)

if ATTEMPT_PATH.exists():
    with open(ATTEMPT_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            attempts_by_id[str(row["id"])].append(row)

print(f"Loaded prior attempts for {len(attempts_by_id)} questions")

def save_responses_atomic():
    tmp_path = RESPONSE_PATH.with_suffix(".tmp")
    backup_path = RESPONSE_PATH.with_suffix(".bak")

    with open(tmp_path, "w") as f:
        for item in data:
            qid = str(item["id"])
            if qid in responses_by_id:
                f.write(json.dumps(responses_by_id[qid]) + "\n")

    if RESPONSE_PATH.exists():
        shutil.copy2(RESPONSE_PATH, backup_path)

    tmp_path.replace(RESPONSE_PATH)

def append_attempt_record(record: dict):
    with open(ATTEMPT_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

    attempts_by_id[str(record["id"])].append(record)

def append_vote_record(record: dict):
    with open(VOTE_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

def summarize_attempts_for_retry(item: dict, max_attempts: int = 12) -> str:
    qid = str(item["id"])
    attempts = attempts_by_id.get(qid, [])[-max_attempts:]

    if not attempts:
        return "No previous attempts were recorded."

    counts = Counter(
        a.get("answer_key", "")
        for a in attempts
        if a.get("answer_key")
    )

    lines = []
    if counts:
        lines.append("Candidate final answers from previous attempts:")
        for ans, count in counts.most_common():
            lines.append(f"- {ans}: {count} vote(s)")
    else:
        lines.append("No previous attempt produced a parsable final answer.")

    lines.append("\nPrevious reasoning/final-output snippets:")
    for i, a in enumerate(attempts, start=1):
        ans = a.get("answer_key", "")
        complete = a.get("complete", False)
        reached = a.get("reached_think_end", False)
        final_text = truncate_text(a.get("final_text", ""), 800)
        raw_preview = truncate_text(a.get("raw_preview", ""), 800)

        lines.append(
            f"\nAttempt {i}: answer_key={ans!r}, complete={complete}, reached_think_end={reached}\n"
            f"Final extracted text:\n{final_text if final_text else '[empty]'}\n"
            f"Raw reasoning/output preview:\n{raw_preview if raw_preview else '[empty]'}"
        )

    return "\n".join(lines)

def make_retry_prompt(item: dict) -> str:
    problem_text = build_problem_text(item)
    previous_summary = summarize_attempts_for_retry(item)

    user = (
        f"Original problem:\n{problem_text}\n\n"
        f"Previous generations disagreed, were incomplete, or did not reach a reliable final answer.\n\n"
        f"{previous_summary}\n\n"
        "Re-evaluate the problem from scratch. "
        "Pay special attention to the conflicting candidate answers above. "
        "Decide which answer is correct, or compute a new answer if all previous candidates are wrong. "
        "End with exactly one final answer in \\boxed{...}, and do not write anything after it."
    )

    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": RETRY_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    # Ensure retry prompt + 81920 output fits inside MAX_MODEL_LEN.
    max_prompt_tokens = max(1024, MAX_MODEL_LEN - RETRY_MAX_TOKENS - 1024)
    ids = tokenizer.encode(prompt)

    if len(ids) > max_prompt_tokens:
        ids = ids[:max_prompt_tokens]
        prompt = tokenizer.decode(ids, skip_special_tokens=True)

    return prompt

def select_majority_response(item: dict, generation_records: List[dict]):
    is_mcq = bool(item.get("options"))

    complete_records = [
        r for r in generation_records
        if r["complete"] and r["answer_key"]
    ]

    if not complete_records:
        return None, {
            "accepted": False,
            "reason": "no_complete_records",
            "vote_counts": {},
            "top_answer": "",
            "top_count": 0,
        }

    counts = Counter(r["answer_key"] for r in complete_records)
    top_answer, top_count = counts.most_common(1)[0]

    if top_count >= AGREE_THRESHOLD:
        candidates = [r for r in complete_records if r["answer_key"] == top_answer]

        # Prefer the shortest complete response among the agreeing candidates.
        # It is usually cleaner for judging.
        best = min(candidates, key=lambda r: len(r["final_text"]))

        return best, {
            "accepted": True,
            "reason": "majority_agreement",
            "vote_counts": dict(counts),
            "top_answer": top_answer,
            "top_count": top_count,
        }

    return None, {
        "accepted": False,
        "reason": "insufficient_agreement",
        "vote_counts": dict(counts),
        "top_answer": top_answer,
        "top_count": top_count,
    }

def save_accepted_response(item: dict, best_record: dict, vote_info: dict, retry_num: int, max_tokens: int):
    qid = str(item["id"])

    responses_by_id[qid] = {
        "id": item["id"],
        "is_mcq": bool(item.get("options")),
        "response": best_record["final_text"],
        "answer_key": best_record["answer_key"],
        "accepted_reason": vote_info["reason"],
        "top_answer": vote_info["top_answer"],
        "top_count": vote_info["top_count"],
        "vote_counts": vote_info["vote_counts"],
        "num_votes": NUM_VOTES,
        "agree_threshold": AGREE_THRESHOLD,
        "retry_num": retry_num,
        "max_tokens": max_tokens,
    }

def generate_votes_for_batch(batch: List[dict], retry_num: int, max_tokens: int):
    if retry_num == 0:
        prompts = [make_initial_prompt(item) for item in batch]
    else:
        prompts = [make_retry_prompt(item) for item in batch]

    sampling_params = make_sampling_params(max_tokens=max_tokens, n=NUM_VOTES)

    outputs = llm.generate(prompts, sampling_params=sampling_params)

    batch_records = {}

    for item, out in zip(batch, outputs):
        qid = str(item["id"])
        is_mcq = bool(item.get("options"))

        records = []

        for vote_idx, completion in enumerate(out.outputs):
            raw_text = completion.text.strip()

            final_text, reached_think_end = extract_qwen_final_content_from_vllm(
                completion=completion,
                raw_text=raw_text,
            )

            answer_key = extract_answer_key(final_text, is_mcq)
            complete = has_complete_answer(final_text, is_mcq)

            record = {
                "id": item["id"],
                "is_mcq": is_mcq,
                "retry_num": retry_num,
                "vote_idx": vote_idx,
                "max_tokens": max_tokens,
                "reached_think_end": reached_think_end,
                "complete": complete,
                "answer_key": answer_key,
                "final_text": final_text,
                "raw_preview": raw_text[:1200],
            }

            append_attempt_record(record)
            records.append(record)

        batch_records[qid] = records

    return batch_records

def run_majority_pipeline():
    total_start = time.time()

    unresolved = [
        item for item in data
        if str(item["id"]) not in responses_by_id
    ]

    print(f"Starting unresolved count: {len(unresolved)}")

    for retry_num in range(MAX_RETRIES + 1):
        if not unresolved:
            break

        max_tokens = DEFAULT_MAX_TOKENS if retry_num == 0 else RETRY_MAX_TOKENS
        label = "initial" if retry_num == 0 else f"retry_{retry_num}"

        print(f"\n=== {label}: {len(unresolved)} unresolved, max_tokens={max_tokens} ===")

        phase_start = time.time()
        next_unresolved = []

        phase_batch_size = BATCH_SIZE if retry_num == 0 else RETRY_BATCH_SIZE
        pbar = tqdm(range(0, len(unresolved), phase_batch_size), desc=label)

        for batch_start in pbar:
            batch = unresolved[batch_start:batch_start + phase_batch_size]

            t0 = time.time()

            try:
                batch_records = generate_votes_for_batch(
                    batch=batch,
                    retry_num=retry_num,
                    max_tokens=max_tokens,
                )

                accepted_this_batch = 0
                unresolved_this_batch = 0

                for item in batch:
                    qid = str(item["id"])
                    records = batch_records[qid]

                    best, vote_info = select_majority_response(item, records)

                    vote_record = {
                        "id": item["id"],
                        "is_mcq": bool(item.get("options")),
                        "retry_num": retry_num,
                        "max_tokens": max_tokens,
                        "accepted": vote_info["accepted"],
                        "reason": vote_info["reason"],
                        "top_answer": vote_info["top_answer"],
                        "top_count": vote_info["top_count"],
                        "vote_counts": vote_info["vote_counts"],
                    }
                    append_vote_record(vote_record)

                    if best is not None:
                        save_accepted_response(
                            item=item,
                            best_record=best,
                            vote_info=vote_info,
                            retry_num=retry_num,
                            max_tokens=max_tokens,
                        )
                        accepted_this_batch += 1
                    else:
                        next_unresolved.append(item)
                        unresolved_this_batch += 1

                save_responses_atomic()

                batch_time = time.time() - t0
                elapsed = time.time() - phase_start
                processed = min(batch_start + len(batch), len(unresolved))
                avg = elapsed / max(processed, 1)
                eta = avg * (len(unresolved) - processed)

                pbar.set_postfix({
                    "accepted_total": f"{len(responses_by_id)}/{len(data)}",
                    "batch_acc": accepted_this_batch,
                    "batch_unres": unresolved_this_batch,
                    "batch_s": f"{batch_time:.1f}",
                    "ETA": format_eta(eta),
                })

            except Exception as e:
                print("\nGeneration crashed. Saving accepted responses before raising error...")
                save_responses_atomic()
                print(f"Saved {len(responses_by_id)} accepted responses to {RESPONSE_PATH}")
                raise e

            finally:
                gc.collect()

        unresolved = next_unresolved

        print(
            f"{label} complete. Accepted so far: {len(responses_by_id)}/{len(data)}. "
            f"Still unresolved: {len(unresolved)}. "
            f"Phase time: {format_eta(time.time() - phase_start)}"
        )

    if unresolved and SAVE_BEST_AFTER_MAX_RETRIES:
        print(f"\nSaving best available responses for {len(unresolved)} unresolved items after max retries...")

        for item in unresolved:
            qid = str(item["id"])
            attempts = attempts_by_id.get(qid, [])

            complete_attempts = [
                a for a in attempts
                if a.get("answer_key") and a.get("complete")
            ]

            if complete_attempts:
                counts = Counter(a["answer_key"] for a in complete_attempts)
                top_answer, top_count = counts.most_common(1)[0]
                candidates = [a for a in complete_attempts if a["answer_key"] == top_answer]
                best = min(candidates, key=lambda a: len(a["final_text"]))

                responses_by_id[qid] = {
                    "id": item["id"],
                    "is_mcq": bool(item.get("options")),
                    "response": best["final_text"],
                    "answer_key": best["answer_key"],
                    "accepted_reason": "best_after_max_retries",
                    "top_answer": top_answer,
                    "top_count": top_count,
                    "vote_counts": dict(counts),
                    "num_votes": NUM_VOTES,
                    "agree_threshold": AGREE_THRESHOLD,
                    "retry_num": MAX_RETRIES,
                    "max_tokens": RETRY_MAX_TOKENS,
                }

        save_responses_atomic()

    print("\nGeneration complete.")
    print(f"Accepted/saved responses: {len(responses_by_id)}/{len(data)}")
    print(f"Responses saved to: {RESPONSE_PATH}")
    print(f"Attempts saved to: {ATTEMPT_PATH}")
    print(f"Vote records saved to: {VOTE_PATH}")
    print(f"Total time: {format_eta(time.time() - total_start)}")

run_majority_pipeline()

responses = [
    responses_by_id[str(item["id"])]["response"]
    for item in data
    if str(item["id"]) in responses_by_id
]

response_ids = [
    item["id"]
    for item in data
    if str(item["id"]) in responses_by_id
]

Cell 3: majority-vote generation with checkpointing...
Using response file: results/vllm_vote_run1_responses.jsonl
Already accepted: 885/1126
Loaded prior attempts for 1126 questions
Starting unresolved count: 241

=== initial: 241 unresolved, max_tokens=32768 ===


initial:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/241 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1205 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

initial complete. Accepted so far: 927/1126. Still unresolved: 199. Phase time: 2.0h

=== retry_1: 199 unresolved, max_tokens=81920 ===


...

Rendering prompts:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/160 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

retry_3 complete. Accepted so far: 1011/1126. Still unresolved: 115. Phase time: 19.7m

Saving best available responses for 115 unresolved items after max retries...

Generation complete.
Accepted/saved responses: 1033/1126
Responses saved to: results/vllm_vote_run1_responses.jsonl
Attempts saved to: results/vllm_vote_run1_attempts.jsonl
Vote records saved to: results/vllm_vote_run1_votes.jsonl
Total time: 4.0h


In [11]:
print("Cell 4: evaluation and score saving...")

t0 = time.time()

# ── Load accepted responses ─────────────────────────────────────
responses_by_id = {}

with open(RESPONSE_PATH, "r") as f:
    for line in f:
        row = json.loads(line)
        responses_by_id[str(row["id"])] = row

print(f"Loaded {len(responses_by_id)}/{len(data)} accepted responses from {RESPONSE_PATH}")

missing = [item["id"] for item in data if str(item["id"]) not in responses_by_id]
if missing:
    print(f"Warning: {len(missing)} responses missing. First few missing ids: {missing[:10]}")

# ── Judger setup ───────────────────────────────────────────────
sys.path.insert(0, ".")
from judger import Judger

judger = Judger(strict_extract=False)

def score_mcq(response: str, gold_letter: str) -> bool:
    pred = extract_answer_key(response, is_mcq=True)
    return pred == gold_letter.strip().upper()

# ── Evaluate ───────────────────────────────────────────────────
results = []

eval_start = time.time()

for item in tqdm(data, desc="Scoring"):
    qid = str(item["id"])

    if qid not in responses_by_id:
        continue

    record = responses_by_id[qid]
    response = record["response"]
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id": item["id"],
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "answer_key": record.get("answer_key", ""),
        "accepted_reason": record.get("accepted_reason", ""),
        "top_count": record.get("top_count", None),
        "vote_counts": record.get("vote_counts", {}),
        "has_boxed": bool(extract_boxed(response)),
        "correct": correct,
    })

eval_time = time.time() - eval_start

# ── Save eval results ──────────────────────────────────────────
with open(EVAL_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

# ── Summary ────────────────────────────────────────────────────
mcq_res = [r for r in results if r["is_mcq"]]
frq_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

summary = {
    "run_name": RUN_NAME,
    "response_path": str(RESPONSE_PATH),
    "attempt_path": str(ATTEMPT_PATH),
    "vote_path": str(VOTE_PATH),
    "eval_path": str(EVAL_PATH),
    "num_total_data": len(data),
    "num_evaluated": len(results),
    "num_missing": len(missing),
    "mcq_correct": sum(r["correct"] for r in mcq_res),
    "mcq_total": len(mcq_res),
    "mcq_accuracy": acc(mcq_res),
    "frq_correct": sum(r["correct"] for r in frq_res),
    "frq_total": len(frq_res),
    "frq_accuracy": acc(frq_res),
    "overall_correct": sum(r["correct"] for r in results),
    "overall_total": len(results),
    "overall_accuracy": acc(results),
    "num_with_boxed": sum(r["has_boxed"] for r in results),
    "num_without_boxed": sum(not r["has_boxed"] for r in results),
    "eval_time_sec": eval_time,
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"MCQ       : {summary['mcq_correct']:4d} / {summary['mcq_total']:4d} = {summary['mcq_accuracy']:.2f}%")
print(f"FRQ       : {summary['frq_correct']:4d} / {summary['frq_total']:4d} = {summary['frq_accuracy']:.2f}%")
print(f"Overall   : {summary['overall_correct']:4d} / {summary['overall_total']:4d} = {summary['overall_accuracy']:.2f}%")
print("-" * 50)
print(f"With boxed answer    : {summary['num_with_boxed']}")
print(f"Without boxed answer : {summary['num_without_boxed']}")
print(f"Missing responses    : {summary['num_missing']}")
print("-" * 50)
print(f"Saved eval to    : {EVAL_PATH}")
print(f"Saved summary to : {SUMMARY_PATH}")
print(f"Eval time        : {eval_time:.2f}s")
print("=" * 50)

print("Cell 4 complete.")

Cell 4: evaluation and score saving...
Loaded 1033/1126 accepted responses from results/vllm_vote_run1_responses.jsonl


Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

EVALUATION RESULTS
MCQ       :  306 /  375 = 81.60%
FRQ       :  365 /  658 = 55.47%
Overall   :  671 / 1033 = 64.96%
--------------------------------------------------
With boxed answer    : 1031
Without boxed answer : 2
Missing responses    : 93
--------------------------------------------------
Saved eval to    : results/vllm_vote_run1_eval.jsonl
Saved summary to : results/vllm_vote_run1_summary.json
Eval time        : 92.87s
Cell 4 complete.
